# Show-o2 1.5B: refusal pilot for understanding
Only **image + text -> text**, with frozen weights. Nine steps following the reference notebook, plus one custom test cell.

Use a Colab GPU (preferably A100; memory use and results remain to be measured). Run cell 1, restart the session, then continue from cell 2.
The branch must be available on GitHub before running the clone in cell 1.

Method: [Arditi et al.](https://arxiv.org/abs/2406.11717), sections 2.3, 2.4 and C.1; [reference code](https://github.com/andyrdt/refusal_direction).
This is a multimodal adaptation: the original 128+128 prompts share one fixed image. The paper does not establish this result for Show-o2; this pilot evaluates it.

In [ ]:
# 1. Clone and install. Run once in a clean runtime.
!git clone --branch experiment/refusal-understanding --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
!git clone https://github.com/andyrdt/refusal_direction.git /content/refusal_direction
!git -C /content/refusal_direction checkout 9d852fae1a9121c78b29142de733cb1340770cc3
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1 pandas openpyxl tqdm
print("Now select Runtime > Restart session. Continue from cell 2.")


In [ ]:
# 2. Load Show-o2 1.5B using the adapter from main.
import sys
import os
import json
import random
import shutil
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

PROJECT = Path('/content/show-o2-space')
UPSTREAM = Path('/content/Show-o')
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / 'show-o2'), '/content/refusal_direction']

from show_o2_space.adapter import ShowO2Adapter, MODEL_ID
from experiments.refusal_utils import (
    UnderstandingInputs, mean_activations, evaluate_candidates, filter_validation,
    addition_hooks, ablation_hooks, generate_response, remove_component,
)

N_BUILD, N_VALIDATION, N_TEST = 128, 32, 10
TOKENS_TO_GENERATE, SEED = 100, 42
torch.manual_seed(SEED)
adapter = ShowO2Adapter()
tokenizer = adapter.tokenizer
N_LAYERS = len(adapter.layers)
print(MODEL_ID, torch.cuda.get_device_name(0), adapter.dtype)
print('Layers:', N_LAYERS, '| Batch: 1 | Decoding: greedy (top_k=1)')


In [ ]:
# 3. Use the same splits and sampling order as the reference notebook.
from dataset.load_dataset import load_dataset_split
rng = random.Random(SEED)
harmful_train = rng.sample(load_dataset_split('harmful', 'train', instructions_only=True), N_BUILD)
harmless_train = rng.sample(load_dataset_split('harmless', 'train', instructions_only=True), N_BUILD)
harmful_val = rng.sample(load_dataset_split('harmful', 'val', instructions_only=True), N_VALIDATION)
harmless_val = rng.sample(load_dataset_split('harmless', 'val', instructions_only=True), N_VALIDATION)

# Both groups share the same image; only the instruction changes.
BUILD_IMAGE = PROJECT / 'inputs/images/example.jpg'
shutil.copy2(UPSTREAM / 'show-o2/docs/mmu/pexels-jane-pham-727419-1571673.jpg', BUILD_IMAGE)
torch.manual_seed(SEED)
inputs = UnderstandingInputs(adapter, BUILD_IMAGE)
OUTPUT = PROJECT / 'outputs/refusal'
OUTPUT.mkdir(parents=True, exist_ok=True)
# Baseline filtering applies only to validation, not the 128 + 128 build prompts.
# I / As follow the reference Qwen proxy; validate these prefixes for Show-o2.
REFUSAL_IDS = [tokenizer.encode(text, add_special_tokens=False)[0] for text in ['I']] #['I', 'As']]
print('Proxy tokens:', [(i, tokenizer.decode([i])) for i in REFUSAL_IDS])
harmful_val_raw, harmless_val_raw = harmful_val, harmless_val
harmful_val, harmless_val, validation_df = filter_validation(
    inputs, harmful_val_raw, harmless_val_raw, REFUSAL_IDS,
)
validation_df.to_json(OUTPUT / 'validation_filter.json', orient='records', indent=2)
display(validation_df.groupby('group').retained.agg(['count', 'sum']).rename(
    columns={'count': 'sampled', 'sum': 'retained'}))
# Inspect retained counts before continuing: both groups must be nonempty.
(OUTPUT / 'data.json').write_text(json.dumps(dict(
    seed=SEED, image=str(BUILD_IMAGE), harmful_train=harmful_train,
    harmless_train=harmless_train, harmful_val=harmful_val, harmless_val=harmless_val,
    harmful_val_raw=harmful_val_raw, harmless_val_raw=harmless_val_raw, refusal_ids=REFUSAL_IDS,
), indent=2), encoding='utf-8')
print('Build:', len(harmful_train), '+', len(harmless_train))
print('Validation after baseline filtering:', len(harmful_val), '+', len(harmless_val))
print('Custom tests: 10 + 10, after step 9.')


In [ ]:
# 4. Inspect the actual input format and post-instruction positions.
_, _, alignment = adapter.prepare_understanding(BUILD_IMAGE, harmless_train[0])
print('Input text (-1 denotes image or time embeddings in the alignment):')
print(tokenizer.decode([t for t in alignment['token_ids'] if t >= 0], skip_special_tokens=False))
print('Multimodal alignment:', alignment)
SOURCE_POSITIONS = inputs.positions
N_POSITIONS = len(SOURCE_POSITIONS)
for position, token_id in zip(SOURCE_POSITIONS, inputs.suffix_ids):
    print(position, token_id, repr(tokenizer.decode([token_id])))
print('Candidates:', N_POSITIONS * N_LAYERS)
print('Source layers to evaluate:', list(range(int(0.8 * N_LAYERS))))


In [ ]:
# 5. Compute mean differences at the INPUT of each decoder block.
harmful_means = mean_activations(inputs, harmful_train)
harmless_means = mean_activations(inputs, harmless_train)
candidate_directions = harmful_means - harmless_means
torch.save(candidate_directions, OUTPUT / 'candidate_directions.pt')
print('Shape [positions, layers, hidden]:', tuple(candidate_directions.shape))
display(pd.DataFrame({
    'harmful_mean': harmful_means[0, 12, :6].numpy(),
    'harmless_mean': harmless_means[0, 12, :6].numpy(),
    'r = harmful - harmless': candidate_directions[0, 12, :6].numpy(),
}))

In [ ]:
# 6. Validation scores; only the first floor(0.8 L) source layers.
# Paper proxy: log(P(refusal) / (1-P(refusal))) at the first answer token.
# Use the baseline-filtered validation sets and proxy tokens from step 3.
selection_df = evaluate_candidates(
    inputs, candidate_directions, harmful_val, harmless_val, REFUSAL_IDS,
)
selection_df.to_json(OUTPUT / 'direction_evaluations.json', orient='records', indent=2)
passed_df = selection_df.query('passes_filters').sort_values('bypass_score')
passed_df.to_json(OUTPUT / 'direction_evaluations_filtered.json', orient='records', indent=2)
display(selection_df)
print('Candidates passing induce > 0 and KL < 0.1:', len(passed_df))
# If none pass, the pilot found no direction satisfying these criteria.
# Scores remain available for inspection; selection uses only passing candidates.
selected = passed_df.iloc[0]
selected_layer, selected_position = int(selected.layer), int(selected.position)
refusal_direction = candidate_directions[SOURCE_POSITIONS.index(selected_position), selected_layer]
print('Selected:', selected.to_dict())


In [ ]:
# 7. Inspect and save the selected direction, preserving its magnitude.
print('Layer:', selected_layer, '| Position:', selected_position)
print('Shape:', refusal_direction.shape, '| Norm:', refusal_direction.norm().item())
print('First components:', refusal_direction[:10])
metadata = dict(
    model=MODEL_ID, seed=SEED, build_per_group=N_BUILD, validation_sampled_per_group=N_VALIDATION,
    validation_harmful_retained=len(harmful_val), validation_harmless_retained=len(harmless_val),
    build_image=str(BUILD_IMAGE), positions=SOURCE_POSITIONS,
    selected_layer=selected_layer, selected_position=selected_position,
    refusal_ids=REFUSAL_IDS, source_capture='decoder block input',
    source_layer_limit=int(0.8 * N_LAYERS), induce_threshold=0.0, kl_threshold=0.1,
    scores=selected.to_dict(),
)
torch.save(dict(direction=refusal_direction, metadata=metadata), OUTPUT / 'direction.pt')
(OUTPUT / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

In [ ]:
# 8. Induce refusal: h' = h + r, at the selected layer and ALL positions.
# Hooks remain active at every autoregressive step of adapter.answer.
ADD_PRE_HOOKS, ADD_OUTPUT_HOOKS = addition_hooks(adapter, refusal_direction, selected_layer)
HARMLESS_EXAMPLE = 'Explain why leaves change color in autumn.'
baseline = generate_response(adapter, BUILD_IMAGE, HARMLESS_EXAMPLE, max_new_tokens=TOKENS_TO_GENERATE)
modified = generate_response(adapter, BUILD_IMAGE, HARMLESS_EXAMPLE,
    pre=ADD_PRE_HOOKS, post=ADD_OUTPUT_HOOKS, max_new_tokens=TOKENS_TO_GENERATE)
print('QUESTION:', HARMLESS_EXAMPLE)
print('BASELINE:', baseline)
print('ADD DIRECTION:', modified)
h = harmless_means[-1, selected_layer]
display(pd.DataFrame({'h_before': h[:6].numpy(), 'r': refusal_direction[:6].numpy(),
                      'h_after': (h + refusal_direction)[:6].numpy()}))
print('Numerical inspection uses the stored mean; actual generated responses are shown above.')


In [ ]:
# 9. Suppress refusal: h' = h - (h dot r_hat) r_hat.
# ALL layers and positions, including image/time and generated tokens.
# Project block inputs and attention/MLP outputs, following the reference.
ABLATION_PRE_HOOKS, ABLATION_OUTPUT_HOOKS = ablation_hooks(adapter, refusal_direction)
HARMFUL_EXAMPLE = harmful_val[0]
baseline = generate_response(adapter, BUILD_IMAGE, HARMFUL_EXAMPLE, max_new_tokens=TOKENS_TO_GENERATE)
modified = generate_response(adapter, BUILD_IMAGE, HARMFUL_EXAMPLE,
    pre=ABLATION_PRE_HOOKS, post=ABLATION_OUTPUT_HOOKS, max_new_tokens=TOKENS_TO_GENERATE)
print('QUESTION:', HARMFUL_EXAMPLE)
print('BASELINE:', baseline)
print('REMOVE COMPONENT:', modified)
h = harmful_means[-1, selected_layer]
h_new = remove_component(h, refusal_direction)
r_hat = refusal_direction / refusal_direction.norm()
display(pd.DataFrame({'h_before': h[:6].numpy(), 'removed_component': (h-h_new)[:6].numpy(),
                      'h_after': h_new[:6].numpy()}))
print('Projection before:', torch.dot(h, r_hat).item())
print('Projection after:', torch.dot(h_new, r_hat).item())


## Custom tests: 10 harmful + 10 harmless
Upload your images through Colab's **Files** panel to the existing `inputs/images/` folder.
Enter ten prompts in each list and their ten corresponding image paths in the same order.
You can use the same ten images for both groups or different paths.
The lists start empty so you can paste your own data. Complete them before running the cell.

Each prompt uses the same image for baseline and intervention. This test set is independent of build/validation.
The Excel file leaves `refusal` empty for manual labels: 1 = refuses, 0 = does not refuse. A non-refusal does not necessarily contain harmful content.
The cell also saves the image/prompt manifest to preserve the pairing.

In [ ]:
# 10. Complete all four lists before running.
harmful_test = [
    # Your 10 harmful prompts, one per line.
]
harmless_test = [
    # Your 10 harmless prompts, one per line.
]
harmful_images = [
    # 'inputs/images/01.jpg', ... enter all 10 paths.
]
harmless_images = [
    # 'inputs/images/01.jpg', ... enter all 10 paths.
]

# DataFrame pairs entries by order and requires equal numbers of prompts and paths.
tests = pd.concat([
    pd.DataFrame(dict(prompt=harmful_test, image=harmful_images)).assign(group='harmful'),
    pd.DataFrame(dict(prompt=harmless_test, image=harmless_images)).assign(group='harmless'),
], ignore_index=True)
tests.to_json(OUTPUT / 'test_data.json', orient='records', indent=2)
conditions = {
    'harmful': ('remove_component', ABLATION_PRE_HOOKS, ABLATION_OUTPUT_HOOKS),
    'harmless': ('add_direction', ADD_PRE_HOOKS, ADD_OUTPUT_HOOKS),
}
rows = []
for prompt_id, item in tqdm(tests.iterrows(), total=len(tests)):
    condition, pre, post = conditions[item.group]
    baseline = generate_response(adapter, item.image, item.prompt, max_new_tokens=TOKENS_TO_GENERATE)
    modified = generate_response(adapter, item.image, item.prompt, pre, post, TOKENS_TO_GENERATE)
    for name, response in [('baseline', baseline), (condition, modified)]:
        rows.append(dict(prompt_id=prompt_id+1, group=item.group, image=item.image,
                         prompt=item.prompt, condition=name, source_layer=selected_layer,
                         source_position=selected_position, response=response, refusal=None))
results_df = pd.DataFrame(rows)
results_df.to_excel(OUTPUT / 'refusal_pilot_results.xlsx', index=False)
display(results_df)
print('Prompts per group:', tests.groupby('group').size().to_dict())
print('With 10 + 10 prompts, 40 responses are saved to', OUTPUT)
from google.colab import files
archive = shutil.make_archive('/content/show-o2-refusal-results', 'zip', OUTPUT)
files.download(archive)